# Model Exploration: Softmax Regression vs. kNN

Loads `train_features.csv` / `val_features.csv` (exported from the cleaning/EDA notebook), trains Softmax Regression and kNN, and reports accuracy, precision/recall/F1, and confusion matrices for each. Also sweeps a key hyperparameter for each model.

- Precision = everything predicted as X (how much was actually X?)
- Recall = everything that was really X (how much did the model catch?)
- F1 = one number combining both

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train = pd.read_csv('train_features.csv')
val = pd.read_csv('val_features.csv')

X_train, y_train = train.drop(columns='Label'), train['Label']
X_val, y_val = val.drop(columns='Label'), val['Label']

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Train: (1017, 22), Val: (218, 22)


## Standardize features

Fit the scaler on train only, then apply the same transform to val (to avoid leakage).

In [2]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

labels = sorted(y_train.unique())
print("Classes:", labels)

Classes: ['Dubai', 'New York City', 'Paris', 'Rio de Janeiro']


## Softmax (Multinomial Logistic) Regression

In [3]:
softmax_model = LogisticRegression(max_iter=1000, random_state=42)
softmax_model.fit(X_train_scaled, y_train)
pred_softmax = softmax_model.predict(X_val_scaled)

acc_softmax = accuracy_score(y_val, pred_softmax)
print(f"Softmax Regression validation accuracy: {acc_softmax:.4f}")
print()
print(classification_report(y_val, pred_softmax))

print("Confusion matrix (rows=true, cols=predicted):")
cm_softmax = confusion_matrix(y_val, pred_softmax, labels=labels)
display(pd.DataFrame(cm_softmax, index=labels, columns=labels))

Softmax Regression validation accuracy: 0.9495

                precision    recall  f1-score   support

         Dubai       0.89      0.93      0.91        55
 New York City       0.98      0.95      0.96        55
         Paris       0.98      0.93      0.95        55
Rio de Janeiro       0.95      1.00      0.97        53

      accuracy                           0.95       218
     macro avg       0.95      0.95      0.95       218
  weighted avg       0.95      0.95      0.95       218

Confusion matrix (rows=true, cols=predicted):


,Dubai,New York City,Paris,Rio de Janeiro
Dubai,51,0,1,3
New York City,3,52,0,0
Paris,3,1,51,0
Rio de Janeiro,0,0,0,53


## k-Nearest Neighbors (k=5 baseline)

In [4]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
pred_knn = knn_model.predict(X_val_scaled)

acc_knn = accuracy_score(y_val, pred_knn)
print(f"kNN (k=5) validation accuracy: {acc_knn:.4f}")
print()
print(classification_report(y_val, pred_knn))

print("Confusion matrix (rows=true, cols=predicted):")
cm_knn = confusion_matrix(y_val, pred_knn, labels=labels)
display(pd.DataFrame(cm_knn, index=labels, columns=labels))

kNN (k=5) validation accuracy: 0.8945

                precision    recall  f1-score   support

         Dubai       0.78      0.93      0.85        55
 New York City       0.94      0.84      0.88        55
         Paris       0.96      0.84      0.89        55
Rio de Janeiro       0.93      0.98      0.95        53

      accuracy                           0.89       218
     macro avg       0.90      0.90      0.90       218
  weighted avg       0.90      0.89      0.89       218

Confusion matrix (rows=true, cols=predicted):


,Dubai,New York City,Paris,Rio de Janeiro
Dubai,51,1,1,2
New York City,7,46,1,1
Paris,6,2,46,1
Rio de Janeiro,1,0,0,52


## Hyperparameter sweep

Tuning k for kNN, and regularization strength C for softmax regression using validation accuracy.

In [ ]:
# kNN
knn_results = []
for k in [3, 5, 7, 9, 11, 15]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_val, m.predict(X_val_scaled))
    knn_results.append({'k': k, 'val_accuracy': round(acc, 4)})

knn_results_df = pd.DataFrame(knn_results)
display(knn_results_df)
best_k = knn_results_df.loc[knn_results_df['val_accuracy'].idxmax(), 'k']
print(f"Best k: {best_k}")

,k,val_accuracy
0,3,0.9037
1,5,0.8945
2,7,0.9266
3,9,0.9358
4,11,0.9358
5,15,0.9037


Best k: 9


In [6]:
# Softmax regression
softmax_results = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(C=C, max_iter=1000, random_state=42)
    m.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_val, m.predict(X_val_scaled))
    softmax_results.append({'C': C, 'val_accuracy': round(acc, 4)})

softmax_results_df = pd.DataFrame(softmax_results)
display(softmax_results_df)
best_C = softmax_results_df.loc[softmax_results_df['val_accuracy'].idxmax(), 'C']
print(f"Best C: {best_C}")

,C,val_accuracy
0,0.01,0.9495
1,0.10,0.9450
2,1.00,0.9495
3,10.00,0.9495
4,100.00,0.9495


Best C: 0.01


## Summary: Softmax Regression vs. kNN

Best result for each model.

In [7]:
summary = pd.DataFrame([
    {'model': 'Softmax Regression (default C=1.0)', 'val_accuracy': round(acc_softmax, 4)},
    {'model': f'Softmax Regression (best C={best_C})', 'val_accuracy': softmax_results_df['val_accuracy'].max()},
    {'model': 'kNN (k=5)', 'val_accuracy': round(acc_knn, 4)},
    {'model': f'kNN (best k={best_k})', 'val_accuracy': knn_results_df['val_accuracy'].max()},
])
display(summary)

,model,val_accuracy
0,Softmax Regression (default C=1.0),0.9495
1,Softmax Regression (best C=0.01),0.9495
2,kNN (k=5),0.8945
3,kNN (best k=9),0.9358


## Summary for Report

I tried two models: softmax regression and kNN.

Before training either model, I standardized all 22 features (scaled based on train only, then applied the same scaling to val/test so there's no leakage). Both models require it for different reasons. For kNN, it's because it works by measuring distance between points, and the features are on different scales — Q7 (temperature) can range, like 65 degrees, but Q6 (word ranks) only goes 1-6. Without scaling, Q7 would dominate every distance calculation and the other features wouldn't matter much. For softmax regression, scaling helps the gradient descent converge properly instead of some features getting larger weights just because their raw numbers are bigger.

For softmax regression, I used the multinomial version that predicts all 4 cities directly in one model, instead of training 4 separate models (one per city) that each only decide whether a response belongs to their one city or not, then picking whichever model was most confident.

For hyperparameters, I tested different C values for softmax regression (0.01, 0.1, 1, 10, 100) and different k values for kNN (3, 5, 7, 9, 11, 15), using validation accuracy to compare. The best C ended up being 0.01, which means more regularization worked better; it prefers a simpler model instead of trying too hard to perfectly fit every training point. That makes sense with what was found in EDA. Some features like Q4 and Q7 already separate the cities clearly on their own, so the model doesn't need to be super complex. For kNN, k=9 worked best, looking at a slightly bigger group of neighbors generalized better than a small one.

For evaluation setup, all models need to be evaluated on the exact same val set, otherwise the models aren't actually being compared fairly. My val set comes from the group-based split, and I used it as-is for both softmax regression and kNN.

For the evaluation metric, I used accuracy as the main number since the 4 classes are balanced (~25% each), so accuracy isn't skewed by one city dominating the dataset. I also looked at precision, recall, and F1 per class, since accuracy alone can hide which specific city a model struggles with. For example, my kNN model mistakenly predicted several Dubai responses as NYC. This was only a small amount of rows out of the whole val set, so it barely moves overall accuracy. But looking at precision and recall per city catches it: NYC's precision drops, since some of what it's calling 'NYC' is actually mislabeled Dubai.